<table align="center">
 <td align="center"><a target="_blank" href="https://colab.research.google.com/github/ezponda/intro_deep_learning/blob/main/class/CNN/OCR_with_EasyOCR_and_TrOCR.ipynb">
        <img src="https://colab.research.google.com/img/colab_favicon_256px.png"  width="50" height="50" style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/OCR_with_EasyOCR_and_TrOCR.ipynb">
        <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"  width="50" height="50" style="padding-bottom:5px;" />View Source on GitHub</a></td>
</table>

# OCR with EasyOCR and TrOCR

## Table of Contents

1. [Introduction](#introduction)
2. [EasyOCR: Full Pipeline](#easyocr-full-pipeline)
3. [Image Preprocessing](#image-preprocessing)
4. [TrOCR: Transformer-Based OCR](#trocr-transformer-based-ocr)
5. [Hybrid Pipeline: EasyOCR Detection + TrOCR Recognition](#hybrid-pipeline)

## Introduction

<a id="introduction"></a>

**Optical Character Recognition (OCR)** converts images of text into machine-readable strings. It sounds simple, but real-world images introduce many challenges:

- **Font variability** — different typefaces, sizes, weights
- **Noise and blur** — low-resolution scans, camera shake, stains
- **Complex layouts** — receipts, invoices, forms with tables and columns
- **Handwriting** — far harder than printed text

### Modern OCR landscape

Traditional OCR systems (like Tesseract) use hand-crafted feature extraction followed by character classification. Modern approaches use deep learning end-to-end:

| Approach | Detection | Recognition | Example |
|----------|-----------|-------------|---------|
| **CRNN-based** | CRAFT network | CNN + BiLSTM + CTC | EasyOCR |
| **Transformer-based** | ViT encoder | Autoregressive decoder | TrOCR |

### Two-stage pipeline

OCR systems typically work in two stages:

1. **Text detection** — find *where* text appears in the image (bounding boxes)
2. **Text recognition** — read *what* each detected region says

In this notebook we will:
- Use **EasyOCR** as a complete pipeline (detection + recognition)
- Apply **image preprocessing** techniques to improve OCR on noisy images
- Use **TrOCR** (a transformer model) for high-quality text recognition
- Build a **hybrid pipeline** combining EasyOCR detection with TrOCR recognition

In [ ]:
#!pip install easyocr
#!pip install transformers torch sentencepiece

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import time

## EasyOCR: Full Pipeline

<a id="easyocr-full-pipeline"></a>

[EasyOCR](https://github.com/JaidedAI/EasyOCR) is a Python library that supports 80+ languages out of the box. It uses:
- **CRAFT** (Character Region Awareness for Text Detection) for finding text regions
- **CRNN** (CNN + BiLSTM + CTC) for recognizing text

Everything installs via pip — no system dependencies needed.

**Multilingual support**: EasyOCR can read multiple languages simultaneously. You specify them when creating the reader:
```python
reader = easyocr.Reader(['en'])              # English only
reader = easyocr.Reader(['en', 'es'])        # English + Spanish
reader = easyocr.Reader(['ch_sim', 'en'])    # Simplified Chinese + English
```
Languages using the same script (e.g., Latin-based) can be combined freely. See the [full list of supported languages](https://www.jaided.ai/easyocr/).

In [ ]:
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/restaurant-bar-receipt-sample.jpg'
urllib.request.urlretrieve(url, 'receipt.jpg')
image = cv2.imread('receipt.jpg')
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 10))
plt.imshow(image_rgb)
plt.axis('off')
plt.title('Receipt image')
plt.show()

In [ ]:
import easyocr

reader = easyocr.Reader(['en'])
results = reader.readtext(image)

### Understanding EasyOCR output

`readtext()` returns a list of tuples, each with three elements:

```
(bbox, text, confidence)
```

- **bbox**: 4 corner points `[[x1,y1], [x2,y2], [x3,y3], [x4,y4]]` (top-left, top-right, bottom-right, bottom-left)
- **text**: the recognized string
- **confidence**: float between 0 and 1

In [ ]:
for (bbox, text, confidence) in results:
    print(f'{text:30s}  (confidence: {confidence:.2f})')

### Decoder options

EasyOCR supports different decoders for the recognition stage:

- `'greedy'` (default) — picks the most likely character at each step. Fast but can miss context.
- `'beamsearch'` — explores multiple hypotheses in parallel. Better accuracy, slower.

The `beamWidth` parameter (default 5) controls how many hypotheses to keep at each step — higher means more thorough but slower.

Similarly, TrOCR supports beam search via `num_beams` in `model.generate()`.

### Confidence filtering

In practice, you often want to discard low-confidence detections. A simple filter:

In [ ]:
# Compare greedy vs beam search decoder
results_greedy = reader.readtext(image, decoder='greedy')
results_beam = reader.readtext(image, decoder='beamsearch', beamWidth=5)

print(f'{"Greedy":<35} {"Beam search":<35}')
print('-' * 70)
for i in range(max(len(results_greedy), len(results_beam))):
    g = results_greedy[i][1] if i < len(results_greedy) else ''
    b = results_beam[i][1] if i < len(results_beam) else ''
    marker = ' ' if g == b else '*'
    print(f'{g:<35} {b:<35} {marker}')

# Filter low-confidence detections
min_confidence = 0.5
filtered = [(bbox, text, conf) for (bbox, text, conf) in results if conf > min_confidence]
print(f'\nAll detections: {len(results)}, after filtering (>{min_confidence}): {len(filtered)}')

In [ ]:
def draw_easyocr_results(image, results, color=(0, 255, 0), thickness=2, font_scale=0.5):
    """Draw bounding boxes and text labels on the image."""
    img_draw = image.copy()
    if len(img_draw.shape) == 2:
        img_draw = cv2.cvtColor(img_draw, cv2.COLOR_GRAY2BGR)

    for item in results:
        # paragraph=True returns (bbox, text), normal returns (bbox, text, confidence)
        bbox, text = item[0], item[1]
        confidence = item[2] if len(item) == 3 else None

        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(img_draw, [pts], isClosed=True, color=color, thickness=thickness)
        x, y = pts[0]
        label = f'{text} ({confidence:.2f})' if confidence is not None else text
        cv2.putText(img_draw, label, (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 1)
    return img_draw

annotated = draw_easyocr_results(image, results)
plt.figure(figsize=(8, 12))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('EasyOCR detections')
plt.show()

### Paragraph mode

By default, EasyOCR returns word-level or line-level detections. Setting `paragraph=True` merges nearby detections into paragraphs, which can be useful for structured documents.

In [ ]:
results_word = reader.readtext(image, paragraph=False)
results_para = reader.readtext(image, paragraph=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 12))

annotated_word = draw_easyocr_results(image, results_word)
axes[0].imshow(cv2.cvtColor(annotated_word, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Word/line mode ({len(results_word)} detections)')
axes[0].axis('off')

annotated_para = draw_easyocr_results(image, results_para, color=(255, 0, 0))
axes[1].imshow(cv2.cvtColor(annotated_para, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Paragraph mode ({len(results_para)} detections)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Question 1:

Use EasyOCR to read the license plate from the image below. Use the `allowlist` parameter to restrict recognition to uppercase letters and digits only.

Hint: `reader.readtext(image, allowlist=...)`

<img src="https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/car_plate.png" width="400">

In [ ]:
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/car_plate.png'
urllib.request.urlretrieve(url, 'car_plate.png')
plate_image = cv2.imread('car_plate.png')

plt.figure(figsize=(6, 3))
plt.imshow(cv2.cvtColor(plate_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
# Read the license plate using allowlist
plate_results = reader.readtext(plate_image, allowlist=...)
for (bbox, text, confidence) in plate_results:
    print(f'{text}  (confidence: {confidence:.2f})')

## Question 2:

Extract the total amount from the bill image below. Use EasyOCR to read all text, then find the line containing the total.

<img src="https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/bill_1.jpg" width="400">

In [ ]:
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/bill_1.jpg'
urllib.request.urlretrieve(url, 'bill.jpg')
bill_image = cv2.imread('bill.jpg')

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(bill_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
# Read all text from the bill
bill_results = reader.readtext(bill_image)
for (bbox, text, confidence) in bill_results:
    print(f'{text:30s}  (confidence: {confidence:.2f})')

In [ ]:
# Extract the total amount from the bill
# Hint: look for keywords like "total", "amount", "sum" in the results
total = ...
print(f'Total amount: {total}')

## Image Preprocessing

<a id="image-preprocessing"></a>

Real-world images are often noisy, poorly lit, or low-contrast. Preprocessing can significantly improve OCR accuracy. We will cover:

1. **Histogram equalization** (CLAHE) — fix contrast
2. **Thresholding** — create clean binary images
3. **Noise removal** — reduce speckles
4. **Morphological operations** — clean up character shapes

In [ ]:
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/bad_quality_receipt.png'
urllib.request.urlretrieve(url, 'bad_receipt.png')
bad_image = cv2.imread('bad_receipt.png', cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6, 8))
plt.imshow(bad_image, cmap='gray')
plt.axis('off')
plt.title('Bad quality receipt')
plt.show()

In [ ]:
# EasyOCR on the raw bad quality image
bad_results = reader.readtext(bad_image)
print('EasyOCR on raw image:')
print('-' * 40)
for (bbox, text, confidence) in bad_results:
    print(f'{text:30s}  ({confidence:.2f})')

### CLAHE (Contrast Limited Adaptive Histogram Equalization)

Standard histogram equalization adjusts the global contrast, but it can over-amplify noise. **CLAHE** divides the image into small tiles and equalizes each one independently, then stitches them with bilinear interpolation. This gives local contrast enhancement without blowing out bright regions.

Key parameters:
- `clipLimit`: threshold for contrast limiting (higher = more contrast, more noise)
- `tileGridSize`: size of the tiles (smaller = more local adaptation)

In [ ]:
def equalize_histogram(image, clip_limit=2.0, tile_size=8):
    """Apply CLAHE histogram equalization."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    return clahe.apply(image)

equalized = equalize_histogram(bad_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(bad_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(equalized, cmap='gray')
axes[1].set_title('CLAHE equalized')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
clahe_results = reader.readtext(equalized)
print('EasyOCR after CLAHE:')
print('-' * 40)
for (bbox, text, confidence) in clahe_results:
    print(f'{text:30s}  ({confidence:.2f})')

### Thresholding

Thresholding converts a grayscale image into a binary (black & white) image. This removes background gradients and makes text stand out clearly.

- **Otsu's method**: automatically finds the optimal threshold value
- **Adaptive thresholding**: computes different thresholds for different regions (useful for uneven lighting)

In [ ]:
def apply_thresholding(image, method='otsu'):
    """Apply thresholding to create a binary image."""
    if method == 'otsu':
        _, result = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif method == 'adaptive_mean':
        result = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                        cv2.THRESH_BINARY, 11, 2)
    elif method == 'adaptive_gaussian':
        result = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 11, 2)
    return result

thresh_otsu = apply_thresholding(equalized, 'otsu')
thresh_adaptive = apply_thresholding(equalized, 'adaptive_gaussian')

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].imshow(equalized, cmap='gray')
axes[0].set_title('CLAHE equalized')
axes[0].axis('off')
axes[1].imshow(thresh_otsu, cmap='gray')
axes[1].set_title("Otsu's thresholding")
axes[1].axis('off')
axes[2].imshow(thresh_adaptive, cmap='gray')
axes[2].set_title('Adaptive Gaussian')
axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
thresh_results = reader.readtext(thresh_otsu)
print('EasyOCR after CLAHE + Otsu thresholding:')
print('-' * 40)
for (bbox, text, confidence) in thresh_results:
    print(f'{text:30s}  ({confidence:.2f})')

### Noise Removal

Median blur replaces each pixel with the median of its neighborhood. Unlike Gaussian blur, it preserves edges well — important for text characters.

In [ ]:
def remove_noise(image, kernel_size=3):
    """Remove noise using median blur."""
    return cv2.medianBlur(image, kernel_size)

denoised = remove_noise(bad_image, kernel_size=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(bad_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(denoised, cmap='gray')
axes[1].set_title('After median blur')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Morphological Operations

Morphological operations use a small structuring element (kernel) to modify shapes in binary images:

- **Dilation**: expands white regions — makes text thicker, fills small gaps
- **Erosion**: shrinks white regions — makes text thinner, removes small noise
- **Opening** (erosion → dilation): removes small white noise while preserving shape
- **Closing** (dilation → erosion): fills small black holes inside characters

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

dilated = cv2.dilate(thresh_otsu, kernel, iterations=1)
eroded = cv2.erode(thresh_otsu, kernel, iterations=1)
opened = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(thresh_otsu, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
images = [thresh_otsu, dilated, eroded, opened, closed]
titles = ['Thresholded', 'Dilated', 'Eroded', 'Opening', 'Closing']
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

def interactive_preprocessing(clip_limit=2.0, tile_size=8,
                               threshold_method='otsu', blur_size=3):
    """Interactive preprocessing pipeline with live OCR."""
    processed = bad_image.copy()

    # Step 1: Noise removal
    if blur_size > 1:
        processed = cv2.medianBlur(processed, blur_size)

    # Step 2: CLAHE
    clahe = cv2.createCLAHE(clipLimit=clip_limit,
                             tileGridSize=(tile_size, tile_size))
    processed = clahe.apply(processed)

    # Step 3: Thresholding
    processed = apply_thresholding(processed, threshold_method)

    # Show result
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(bad_image, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title('Preprocessed')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Run OCR
    results = reader.readtext(processed)
    print('OCR results:')
    print('-' * 40)
    for (bbox, text, confidence) in results:
        print(f'{text:30s}  ({confidence:.2f})')

interact(interactive_preprocessing,
         clip_limit=FloatSlider(min=1.0, max=10.0, step=0.5, value=2.0),
         tile_size=IntSlider(min=2, max=16, step=2, value=8),
         threshold_method=Dropdown(options=['otsu', 'adaptive_mean', 'adaptive_gaussian'],
                                    value='otsu'),
         blur_size=Dropdown(options=[1, 3, 5, 7], value=3));

## Question 3:

Find the best preprocessing pipeline for the bad quality receipt. Combine techniques (noise removal, CLAHE, thresholding, morphological operations) to maximize OCR accuracy.

Compare the raw OCR output with your best preprocessed result.

In [ ]:
# Build your best preprocessing pipeline
def best_preprocessing(image):
    processed = image.copy()
    # Step 1: ...
    # Step 2: ...
    # Step 3: ...
    return processed

preprocessed = best_preprocessing(bad_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(bad_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(preprocessed, cmap='gray')
axes[1].set_title('Your preprocessing')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Compare OCR results
best_results = reader.readtext(preprocessed)
print('OCR after your preprocessing:')
print('-' * 40)
for (bbox, text, confidence) in best_results:
    print(f'{text:30s}  ({confidence:.2f})')

## TrOCR: Transformer-Based OCR

<a id="trocr-transformer-based-ocr"></a>

[TrOCR](https://arxiv.org/abs/2109.10282) (Transformer-based OCR) is a model from Microsoft that treats OCR as an **image-to-sequence** problem — just like image captioning, but the "caption" is the text in the image.

### Architecture

TrOCR uses an encoder-decoder architecture:

1. **Encoder**: A Vision Transformer (initialized from BEiT) processes the input image into feature embeddings
2. **Decoder**: A text transformer (initialized from RoBERTa) autoregressively generates the output text, token by token

Unlike CRNN-based methods (which use CTC loss), TrOCR generates text **autoregressively** — each token is conditioned on all previous tokens. This makes it better at handling context and complex scripts.

**Important**: TrOCR is a **recognition-only** model. It expects a cropped image of a single text line. It does not do text detection.

### Model variants

| Model | Size | Training data | Best for |
|-------|------|--------------|----------|
| `trocr-small-printed` | ~120M params | Synthetic printed text | Printed text, receipts, documents |
| `trocr-base-printed` | ~330M params | Synthetic printed text | Higher accuracy on printed text |
| `trocr-small-handwritten` | ~120M params | IAM Handwriting Database | Handwritten notes, forms |
| `trocr-base-handwritten` | ~330M params | IAM Handwriting Database | Higher accuracy on handwriting |

We will use the `small` variants since they are faster and fit comfortably on Colab's free GPU.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

# Load the printed text model
processor_printed = TrOCRProcessor.from_pretrained('microsoft/trocr-small-printed')
model_printed = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-printed')

def trocr_predict(image_pil, processor, model):
    """Run TrOCR on a single text line image."""
    pixel_values = processor(images=image_pil, return_tensors='pt').pixel_values
    generated_ids = model.generate(pixel_values)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

print('TrOCR printed model loaded.')

In [ ]:
# Crop a single text line from the receipt using EasyOCR bounding boxes
# We take one of the detected regions
bbox = results[0][0]  # first detection's bounding box
pts = np.array(bbox, dtype=np.int32)

# Get bounding rectangle
x_min, y_min = pts.min(axis=0)
x_max, y_max = pts.max(axis=0)

# Add small padding
pad = 5
x_min, y_min = max(0, x_min - pad), max(0, y_min - pad)
x_max, y_max = min(image.shape[1], x_max + pad), min(image.shape[0], y_max + pad)

# Crop and convert to PIL
cropped = image_rgb[y_min:y_max, x_min:x_max]
cropped_pil = Image.fromarray(cropped)

plt.figure(figsize=(8, 2))
plt.imshow(cropped)
plt.axis('off')
plt.title('Cropped text line')
plt.show()

# Run TrOCR
trocr_text = trocr_predict(cropped_pil, processor_printed, model_printed)
easyocr_text = results[0][1]
print(f'EasyOCR read:  "{easyocr_text}"')
print(f'TrOCR read:    "{trocr_text}"')

### TrOCR on handwritten text

TrOCR really shines on handwritten text, where traditional OCR systems struggle. The handwritten model was trained on the [IAM Handwriting Database](https://fki.tic.heia-fr.ch/databases/iam-handwriting-database).

Let's load a handwritten text sample and compare the printed vs handwritten models.

In [ ]:
# Load handwritten model
processor_handwritten = TrOCRProcessor.from_pretrained('microsoft/trocr-small-handwritten')
model_handwritten = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-handwritten')

# Load IAM handwritten sample
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/iam_handwritten_sample.jpg'
urllib.request.urlretrieve(url, 'handwritten.jpg')
hw_image = Image.open('handwritten.jpg').convert('RGB')

plt.figure(figsize=(10, 3))
plt.imshow(hw_image)
plt.axis('off')
plt.title('IAM handwritten sample')
plt.show()

# Compare printed vs handwritten model
text_printed = trocr_predict(hw_image, processor_printed, model_printed)
text_handwritten = trocr_predict(hw_image, processor_handwritten, model_handwritten)

print(f'Printed model read:      "{text_printed}"')
print(f'Handwritten model read:  "{text_handwritten}"')

## Question 4:

Crop a different text line from the receipt and compare the printed TrOCR model with the EasyOCR result. Then try the **handwritten** model on the same printed text line.

What happens when you use the wrong model variant?

In [ ]:
# Pick a different detection (change the index)
idx = ...
bbox = results[idx][0]
pts = np.array(bbox, dtype=np.int32)

x_min, y_min = pts.min(axis=0)
x_max, y_max = pts.max(axis=0)
pad = 5
x_min, y_min = max(0, x_min - pad), max(0, y_min - pad)
x_max, y_max = min(image.shape[1], x_max + pad), min(image.shape[0], y_max + pad)

cropped = image_rgb[y_min:y_max, x_min:x_max]
cropped_pil = Image.fromarray(cropped)

plt.figure(figsize=(8, 2))
plt.imshow(cropped)
plt.axis('off')
plt.show()

# Compare EasyOCR, TrOCR printed, and TrOCR handwritten
easyocr_text = results[idx][1]
printed_text = trocr_predict(cropped_pil, processor_printed, model_printed)
handwritten_text = trocr_predict(cropped_pil, processor_handwritten, model_handwritten)

print(f'EasyOCR (CRNN):           "{easyocr_text}"')
print(f'TrOCR (printed model):    "{printed_text}"')
print(f'TrOCR (handwritten model): "{handwritten_text}"')

## Hybrid Pipeline: EasyOCR Detection + TrOCR Recognition

<a id="hybrid-pipeline"></a>

We can combine the best of both worlds:

- **EasyOCR's CRAFT detector** for finding text regions (it's fast and accurate)
- **TrOCR** for reading each detected region (higher quality recognition)

EasyOCR's `detect()` method returns only bounding boxes without running recognition, which is much faster.

In [ ]:
# Use EasyOCR for detection only
horizontal_list, free_list = reader.detect(image)

# horizontal_list contains [x_min, x_max, y_min, y_max] for each text region
print(f'Detected {len(horizontal_list[0])} horizontal text regions')
print(f'Detected {len(free_list[0])} free-form text regions')
print()
print('First 5 horizontal boxes (x_min, x_max, y_min, y_max):')
for box in horizontal_list[0][:5]:
    print(f'  {box}')

In [ ]:
def hybrid_ocr(image, reader, processor, model, pad=5, is_bgr=True):
    """Run hybrid pipeline: EasyOCR detection + TrOCR recognition.

    Args:
        is_bgr: Set True if image comes from cv2.imread (BGR), False if already RGB or grayscale.
    """
    if len(image.shape) == 2:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif is_bgr:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image

    horizontal_list, free_list = reader.detect(image)
    results = []

    for box in horizontal_list[0]:
        x_min, x_max, y_min, y_max = box
        # Add padding
        x_min = max(0, x_min - pad)
        y_min = max(0, y_min - pad)
        x_max = min(image_rgb.shape[1], x_max + pad)
        y_max = min(image_rgb.shape[0], y_max + pad)

        cropped = image_rgb[y_min:y_max, x_min:x_max]
        cropped_pil = Image.fromarray(cropped)
        text = trocr_predict(cropped_pil, processor, model)

        # Convert box to corner format for visualization
        bbox = [[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]]
        results.append((bbox, text, 1.0))

    return results

In [ ]:
# Run both pipelines with timing
t0 = time.time()
easyocr_results = reader.readtext(image)
easyocr_time = time.time() - t0

t0 = time.time()
hybrid_results = hybrid_ocr(image, reader, processor_printed, model_printed)
hybrid_time = time.time() - t0

# Show side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 12))

annotated_easy = draw_easyocr_results(image, easyocr_results)
axes[0].imshow(cv2.cvtColor(annotated_easy, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'EasyOCR ({easyocr_time:.1f}s)')
axes[0].axis('off')

annotated_hybrid = draw_easyocr_results(image, hybrid_results, color=(255, 0, 0))
axes[1].imshow(cv2.cvtColor(annotated_hybrid, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Hybrid: CRAFT + TrOCR ({hybrid_time:.1f}s)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Comparison table
print(f'{"":<30} {"EasyOCR":<30} {"Hybrid (TrOCR)":<30}')
print('-' * 90)
max_len = max(len(easyocr_results), len(hybrid_results))
for i in range(max_len):
    easy_text = easyocr_results[i][1] if i < len(easyocr_results) else ''
    hybrid_text = hybrid_results[i][1] if i < len(hybrid_results) else ''
    match = 'Y' if easy_text.strip() == hybrid_text.strip() else ' '
    print(f'[{match}] {"":<27} {easy_text:<30} {hybrid_text:<30}')

print()
print(f'EasyOCR time:  {easyocr_time:.2f}s')
print(f'Hybrid time:   {hybrid_time:.2f}s')

## Question 5:

Run a three-way comparison on the bad quality receipt:

1. **Raw EasyOCR** — no preprocessing
2. **Preprocessed EasyOCR** — use your best preprocessing from Question 3
3. **Hybrid pipeline** — EasyOCR detection + TrOCR recognition (with preprocessing)

Which approach gives the best results? Does preprocessing help the hybrid pipeline too?

In [ ]:
# 1. Raw EasyOCR
raw_results = reader.readtext(bad_image)

# 2. Preprocessed EasyOCR
preprocessed = ...  # use your best_preprocessing function from Q3
preprocessed_results = reader.readtext(preprocessed)

# 3. Hybrid pipeline on preprocessed image
hybrid_bad_results = hybrid_ocr(preprocessed, reader, processor_printed, model_printed)

# Compare all three
print('=== Raw EasyOCR ===')
for (_, text, conf) in raw_results:
    print(f'  {text:30s}  ({conf:.2f})')

print()
print('=== Preprocessed EasyOCR ===')
for (_, text, conf) in preprocessed_results:
    print(f'  {text:30s}  ({conf:.2f})')

print()
print('=== Hybrid (preprocessed + TrOCR) ===')
for (_, text, _) in hybrid_bad_results:
    print(f'  {text}')

## What's Next?

In this notebook we covered the two main deep learning approaches to OCR: CRNN-based (EasyOCR) and Transformer-based (TrOCR). Here are some directions to explore further:

- **[PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR)**: A very complete OCR system. Its **PP-StructureV3** pipeline includes layout detection, table recognition, and structured document parsing — ideal for invoices and forms in production.
- **[docTR](https://github.com/mindee/doctr)**: A modular detection + recognition library that supports bounding boxes, polygons, and binary segmentation. Clean Python API.
- **[Donut](https://github.com/clovaai/donut)** (Document Understanding Transformer): An OCR-free model that directly understands documents without separate detection/recognition stages.
- **Vision-Language Models**: Models like GPT-4V and Gemini can extract text from images as part of their general capabilities — no specialized OCR pipeline needed.
- **Fine-tuning**: Both EasyOCR and TrOCR can be fine-tuned on custom datasets (specific fonts, handwriting styles, or domain-specific text). See [this TrOCR fine-tuning tutorial](https://github.com/NielsRogge/Transformers-Tutorials/tree/master/TrOCR).